In [293]:
import pandas as pd
import pronto
from collections import deque
from typing import Tuple, Dict, Set, List


In [295]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)


In [372]:
df_clinical = pd.read_csv("./data/mapped_all/mapped_clinical_data.csv", dtype=str)
df_preclinical = pd.read_csv("./data/mapped_all/mapped_preclinical_data_enriched.csv", dtype=str)


In [374]:
df_preclinical[df_preclinical['PMID']=="29526407"]

,PMID,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,linkbert_mapped_conditions,linkbert_mapped_drugs,disease_term_mondo_norm,disease_mondo_termid,drug_term_umls_norm,drug_umls_termid
545923,29526407,multiple sclerosis,murine anti-cd52 antibody|anti-cd52 antibody|a...,multiple sclerosis,murine anti-cd52 antibody|anti-cd52 antibody|a...,multiple sclerosis,MONDO:0005301,murine anti-cd52 antibody|anti-cd52 antibody|a...,-1|-1|-1


In [376]:
df_clinical.shape, df_preclinical.shape

((18609, 8), (547365, 9))

In [298]:
%%capture
ontology = pronto.Ontology("data/mondo/mondo.owl")


In [300]:
term = ontology["MONDO:0858950"]  # obsolete 46,XX sex reversal

# Get all superclasses (i.e., recursive parent terms)
print("All ancestor terms:")
for parent in term.superclasses():
    print(parent.id, "-", parent.name)

All ancestor terms:
MONDO:0858950 - traumatic brain injury
MONDO:0043510 - brain injury
MONDO:0005560 - brain disorder
MONDO:0044745 - nervous system injury
MONDO:0002602 - central nervous system disorder
MONDO:0005071 - nervous system disorder
MONDO:0021178 - injury
MONDO:0700096 - human disease
MONDO:0000001 - disease
BFO:0000016 - disposition
BFO:0000017 - realizable entity
BFO:0000020 - specifically dependent continuant
BFO:0000002 - continuant
BFO:0000001 - entity


In [301]:
# 2) Build the full set of valid Mondo IDs in your dataset (skip '-1')

all_mondo_ids_clinical = {
    tid
    for cell in df_clinical['disease_mondo_termid']
    for tid in cell.split('|')
    if tid and tid != '-1'
}

all_mondo_ids_preclinical = {
    tid
    for cell in df_preclinical['disease_mondo_termid']
    for tid in cell.split('|')
    if tid and tid != '-1'
}

all_mondo_ids = all_mondo_ids_clinical.union(all_mondo_ids_preclinical)

# 3) Precompute for each dataset ID:
# ancestor_map lets you quickly ask “what are all the ancestor MONDO terms of X?”
# distance_map lets you ask “how many is_a steps separate term X from ancestor Y?”

ancestor_map = {}
distance_map = {}

for mid in all_mondo_ids:
    try:
        term = ontology[mid]
    except KeyError:
        continue

    # BFS to get all superclasses with distances
    visited = {mid: 0}
    queue = [(term, 0)]
    while queue:
        node, dist = queue.pop(0)
        for parent in node.superclasses(distance=1):
            pid = parent.id
            if "MONDO:" not in pid:
                continue
            if pid not in visited:
                visited[pid] = dist + 1
                queue.append((parent, dist + 1))

    # store ancestors (excluding itself) and distances
    ancestor_map[mid] = set(visited) - {mid}
    for pid, dist in visited.items():
        if pid != mid:
            distance_map[(mid, pid)] = dist

In [302]:
def compute_mondo_term_metrics(
    ontology,
    root_id="MONDO:0000001",
    term_ids=None
):
    """
    Compute two key graph metrics for MONDO terms:
      1. depth_to_root: the shortest number of `is_a` hops from each term up to the chosen root.
      2. desc_count:    the total number of transitive `is_a` descendants under each term.

    Parameters
    ----------
    ontology : pronto.Ontology
        A loaded MONDO ontology (e.g. pronto.Ontology("data/mondo.owl")).
    root_id : str, optional
        The MONDO identifier to treat as the “root” of your specificity hierarchy.
        Defaults to "MONDO:0000001" (“disease or disorder”).
    term_ids : iterable of str, optional
        A collection of MONDO term IDs to score. If None, every ontology term
        whose ID starts with "MONDO:" will be included (excluding obsoletes).

    Returns
    -------
    depth_to_root : dict[str, int]
        Maps each term ID → minimum number of `is_a` edges up to `root_id`.
        Terms unreachable from the root get float('inf').
    desc_count : dict[str, int]
        Maps each term ID → total count of all transitive subclasses under it.

    Example
    -------
    >>> from pronto import Ontology
    >>> ont = Ontology("data/mondo.owl")
    >>> my_ids = ["MONDO:0005301", "MONDO:0000568"]
    >>> depth, descs = compute_mondo_term_metrics(ont, term_ids=my_ids)
    >>> depth["MONDO:0005301"]
    8
    >>> descs["MONDO:0000568"]
    123

    """
    # 1) Build the list of candidates
    if term_ids is None:
        term_ids = [t.id for t in ontology.terms()
                    if t.id.startswith("MONDO:") and not t.obsolete]

    root = ontology[root_id]

    depth_to_root = {}
    desc_count    = {}

    for tid in term_ids:
        # Safely skip any missing/obsolete IDs
        try:
            term = ontology[tid]
        except KeyError:
            continue

        # ---- A) BFS upward to compute depth_to_root ----
        visited = {tid: 0}
        queue   = deque([(term, 0)])
        d_root  = None

        while queue:
            node, dist = queue.popleft()

            # Found the root: record and stop searching
            if node.id == root_id:
                d_root = dist
                break

            # Otherwise enqueue each direct superclass
            for parent in node.superclasses(distance=1, with_self=False):
                pid = parent.id
                if pid not in visited:
                    visited[pid] = dist + 1
                    queue.append((parent, dist + 1))

        depth_to_root[tid] = d_root if d_root is not None else float("inf")

        # ---- B) BFS downward to compute desc_count ----
        descendants = set()
        queue = deque([term])

        while queue:
            node = queue.popleft()
            for child in node.subclasses(distance=1, with_self=False):
                cid = child.id
                if cid not in descendants:
                    descendants.add(cid)
                    queue.append(child)

        desc_count[tid] = len(descendants)

    return depth_to_root, desc_count


In [303]:
candidates = set(ancestor_map.keys()) | set().union(*ancestor_map.values())

depth_to_root, desc_count = compute_mondo_term_metrics(
    ontology,
    root_id="MONDO:0000001",
    term_ids=candidates  # or omit to process every live MONDO term
)

In [304]:
generic_blacklist = {
    "MONDO:0005560",  # brain disorder
    "MONDO:0005084",  # mental disorder
    "MONDO:0005395",  # movement disorder
}

In [305]:
depth_to_root['MONDO:0005027'], depth_to_root['MONDO:0005084'], depth_to_root['MONDO:0005395']

(5, 3, 3)

In [306]:
depth_to_root['MONDO:0021178']


inf

In [307]:
desc_count['MONDO:0021178'], desc_count['MONDO:0005084'], desc_count['MONDO:0005303']

(23, 532, 12)

In [308]:
best_ancestor = None
best_distance = None
child_id = "MONDO:0858950"
min_depth = 5
max_desc = 500

# Examine every precomputed ancestor
for ancestor_id in ancestor_map.get(child_id, []):
    print(ancestor_id, ontology[ancestor_id].name)
    # 1) Must appear elsewhere in the dataset
    if ancestor_id not in all_mondo_ids:
        continue

    # 2) Must pass the depth-to-root filter (too generic if shallow)
    depth = depth_to_root.get(ancestor_id, float("inf"))
    if not (min_depth <= depth < float("inf")):
        # either too shallow or never reached the root
        continue

    # 3) Must pass the descendant-count filter (too broad if many descendants)
    if desc_count.get(ancestor_id, 0) >= max_desc:
        continue

    # 4) Score by how “near” the ancestor is to the child
    dist = distance_map.get((child_id, ancestor_id), float("inf"))

    # Keep the candidate with the smallest hop count
    if best_distance is None or dist > best_distance:
        best_distance = dist
        best_ancestor = ancestor_id
        
best_ancestor_name = ontology[best_ancestor].name if best_ancestor else "NONE"        
print(f"best ancestor: {best_ancestor_name}")

MONDO:0002602 central nervous system disorder
MONDO:0021178 injury
MONDO:0044745 nervous system injury
MONDO:0043510 brain injury
MONDO:0000001 disease
MONDO:0005560 brain disorder
MONDO:0700096 human disease
MONDO:0005071 nervous system disorder
best ancestor: NONE


In [309]:

def assign_nearest_dataset_parents(
    df: pd.DataFrame,
    ontology: pronto.Ontology,
    all_mondo_ids: Set[str],
    ancestor_map: Dict[str, Set[str]],
    depth_to_root: Dict[str, int],
    desc_count: Dict[str, int],
    distance_map: Dict[Tuple[str, str], int],
    min_depth: int = 5,
    max_desc: int = 500,
    id_column: str = "disease_mondo_termid"
) -> pd.DataFrame:
    """
    For each row in `df`, collapse each input MONDO term to its nearest “dataset parent”
    subject to genericity filters. Returns a new DataFrame with two added columns:
      - `nearest_dataset_parent_mondo`
      - `nearest_dataset_parent_label`

    Parameters
    ----------
    df : pd.DataFrame
        Input data frame containing a column of pipe-separated MONDO IDs.
    ontology : pronto.Ontology
        Loaded MONDO ontology for term lookups (`ontology[tid].name`).
    all_mondo_ids : Set[str]
        All MONDO IDs observed across the dataset (to enforce dataset consistency).
    ancestor_map : Dict[str, Set[str]]
        Maps each MONDO ID → its set of all transitive ancestor IDs.
    depth_to_root : Dict[str, int]
        Maps each MONDO ID → number of `is_a` hops to the ontology root.
    desc_count : Dict[str, int]
        Maps each MONDO ID → count of all transitive subclasses under it.
    distance_map : Dict[(str, str), int]
        Maps `(child_id, ancestor_id)` → number of hops between them.
    min_depth : int, default=5
        Minimum allowed depth_to_root for a valid parent (filters out too-generic terms).
    max_desc : int, default=500
        Maximum allowed desc_count for a valid parent (filters out too-broad terms).
    id_column : str, default="disease_mondo_termid"
        Name of the DataFrame column containing pipe-separated MONDO terms.

    Returns
    -------
    pd.DataFrame
        A copy of `df` with two new columns:
          - `nearest_dataset_parent_mondo`: chosen MONDO ID(s) or "-1"
          - `nearest_dataset_parent_label`: corresponding label(s) or "-1"
    """
    parent_ids: list[str] = []
    parent_labels: list[str] = []

    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Parse input IDs (split on '|' and skip placeholders)
        input_ids = [tid for tid in row[id_column].split("|") if tid and tid != "-1"]
        row_parents: list[str] = []
        row_labels: list[str] = []

        # For each input term, find the best ancestor
        for child_id in input_ids:
            best_ancestor = None
            best_distance = None

            # Examine every precomputed ancestor
            for ancestor_id in ancestor_map.get(child_id, []):
                # 1) Must appear elsewhere in the dataset
                if ancestor_id not in all_mondo_ids:
                    continue

                # 2) Must pass the depth-to-root filter (too generic if shallow)
                    # 2) Must pass the depth-to-root filter (too generic if shallow)
                depth = depth_to_root.get(ancestor_id, float("inf"))
                if not (min_depth <= depth < float("inf")):
                    # either too shallow or never reached the root
                    continue

                # 3) Must pass the descendant-count filter (too broad if many descendants)
                if desc_count.get(ancestor_id, 0) >= max_desc:
                    continue

                # 4) Score by how “near” the ancestor is to the child
                dist = distance_map.get((child_id, ancestor_id), float("inf"))

                # Keep the candidate with the smallest hop count
                if best_distance is None or dist > best_distance:
                    best_distance = dist
                    best_ancestor = ancestor_id

            # Record the chosen ancestor (or "-1" if none matched)
            if best_ancestor:
                row_parents.append(best_ancestor)
                row_labels.append(ontology[best_ancestor].name)
            else:
                row_parents.append("-1")
                row_labels.append("-1")

        # Join multiple parents/labels with '|' or default to "-1"
        parent_ids.append("|".join(row_parents) if row_parents else "-1")
        parent_labels.append("|".join(row_labels) if row_labels else "-1")

    # Attach results to a copy of the DataFrame
    result = df.copy()
    result["nearest_dataset_parent_mondo"] = parent_ids
    result["nearest_dataset_parent_label"] = parent_labels

    return result


## Perform mapping preclinical/ clinical

In [378]:
df_expanded_preclinical = assign_nearest_dataset_parents(
    df_preclinical,
    ontology,
    all_mondo_ids,
    ancestor_map,
    depth_to_root,
    desc_count,
    distance_map,
    min_depth = 5,
    max_desc = 500,
    id_column = "disease_mondo_termid"
)

In [379]:
df_expanded_clinical = assign_nearest_dataset_parents(
    df_clinical,
    ontology,
    all_mondo_ids,
    ancestor_map,
    depth_to_root,
    desc_count,
    distance_map,
    min_depth = 5,
    max_desc = 500,
    id_column = "disease_mondo_termid"
)

In [380]:
def merge_original_and_parent_mondo(
    df: pd.DataFrame,
    id_col: str = "disease_mondo_termid",
    label_col: str = "disease_term_mondo_norm",
    parent_id_col: str = "nearest_dataset_parent_mondo",
    parent_label_col: str = "nearest_dataset_parent_label",
    merged_id_col: str = "merged_mondo_termid",
    merged_label_col: str = "merged_mondo_label"
) -> pd.DataFrame:
    """
    For each row:
      1. Split `id_col` and `label_col` into parallel lists orig_ids, orig_labels.
      2. Split parent columns into parent_ids, parent_labels.
      3. Keep orig_ids & orig_labels exactly (including '-1' slots).
      4. Append any parent_id != '-1' that isn’t already in orig_ids, 
         along with its matching parent_label.
      5. Re-join into pipe-delimited merged_id_col / merged_label_col.
    """
    df = df.copy()
    merged_ids = []
    merged_labels = []

    for _, row in df.iterrows():
        # 1) Original slots
        orig_ids     = row[id_col].split("|")
        orig_labels  = row[label_col].split("|")

        # 2) Parent slots
        parent_ids    = row[parent_id_col].split("|")
        parent_labels = row[parent_label_col].split("|")

        # 3) Start merged with *all* original slots
        mids   = list(orig_ids)
        mlabs  = list(orig_labels)

        # 4) Append new parents (drop "-1" and duplicates)
        for pid, plab in zip(parent_ids, parent_labels):
            if pid != "-1" and pid not in orig_ids:
                mids.append(pid)
                mlabs.append(plab)

        # 5) Join back
        merged_ids.append("|".join(mids))
        merged_labels.append("|".join(mlabs))

    df[merged_id_col]    = merged_ids
    df[merged_label_col] = merged_labels
    return df

In [381]:
df_final_preclinical = merge_original_and_parent_mondo(
    df_expanded_preclinical,
    id_col="disease_mondo_termid",
    label_col="disease_term_mondo_norm",
    parent_id_col="nearest_dataset_parent_mondo",
    parent_label_col="nearest_dataset_parent_label"
)

In [382]:
df_final_preclinical

,PMID,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,linkbert_mapped_conditions,linkbert_mapped_drugs,disease_term_mondo_norm,disease_mondo_termid,drug_term_umls_norm,drug_umls_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label
0,31733831,asthma,isorhynchophylline,asthma,isorhynchophylline,asthma,MONDO:0004979,isorhynchophylline,C0245133,-1,-1,MONDO:0004979,asthma
1,31733833,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,MONDO:0005068,antgomir-1192|mir-1192|agomir-1192,-1|-1|-1,-1,-1,MONDO:0005068,myocardial infarction
2,31733925,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,MONDO:0007915,g2|HLA-G2 Isoform,-1|C0967254,-1,-1,MONDO:0007915,systemic lupus erythematosus
3,31733940,cognitive impairment,minocycline,cognitive impairment,minocycline,cognitive disorder,MONDO:0002039,Minocycline,C0026187,-1,-1,MONDO:0002039,cognitive disorder
4,31734027,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620,Tadalafil|Tadalafil,C1176316|C1176316,-1,-1,-1|-1|MONDO:0003620,oxaliplatin-induced peripheral neuropathy|cumu...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
547360,39278849,multiple sclerosis,simvastatin|ag490,multiple sclerosis,simvastatin|ag490,multiple sclerosis,MONDO:0005301,Simvastatin|AG-490,C0074554|C0381241,-1,-1,MONDO:0005301,multiple sclerosis
547361,2434820,parkinson disease,ethanol|acetaldehyde,parkinson disease,ethanol|acetaldehyde,Parkinson disease,MONDO:0005180,Ethanol|Acetaldehyde,C0001962|C0000966,MONDO:0003996,basal ganglia disorder,MONDO:0005180|MONDO:0003996,Parkinson disease|basal ganglia disorder
547362,3300471,multiple sclerosis,met-enkephalin|enkephalins|leu-enkephalin,multiple sclerosis,met-enkephalin|enkephalins|leu-enkephalin,multiple sclerosis,MONDO:0005301,"Enkephalin, Methionine|Enkephalins|Leucine enk...",C0014295|C0014298|C0014294,-1,-1,MONDO:0005301,multiple sclerosis
547363,25052192,multiple sclerosis,modulators of estrogen receptors|selective est...,multiple sclerosis,modulators of estrogen receptors|selective est...,multiple sclerosis,MONDO:0005301,modulators of estrogen receptors|Selective est...,-1|C0732611|C0014912,-1,-1,MONDO:0005301,multiple sclerosis


In [383]:
df_final_clinical = merge_original_and_parent_mondo(
    df_expanded_clinical,
    id_col="disease_mondo_termid",
    label_col="disease_term_mondo_norm",
    parent_id_col="nearest_dataset_parent_mondo",
    parent_label_col="nearest_dataset_parent_label"
)

In [384]:
df_final_clinical

,nct_id,linkbert_aact_mapped_conditions,linkbert_aact_mapped_drugs,Disease Class,drug_term_umls_norm,drug_umls_termid,disease_term_mondo_norm,disease_mondo_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label
0,NCT00000307,alcohol-related disorders|alcoholic cocaine de...,naltrexone,Diseases Category,Naltrexone,C0027360,alcohol-related disorders|alcoholic cocaine de...,MONDO:0021698|-1|MONDO:0005186,-1|MONDO:0005303,-1|drug dependence,MONDO:0021698|-1|MONDO:0005186|MONDO:0005303,alcohol-related disorders|alcoholic cocaine de...
1,NCT00000333,cocaine|cocaine-related disorders,atropine|benzatropine|benztropine|da reuptake ...,Diseases Category,Atropine|Benztropine|Benztropine|da reuptake i...,C0004259|C0005098|C0005098|-1,cocaine|cocaine dependence,-1|MONDO:0005186,MONDO:0005303,drug dependence,-1|MONDO:0005186|MONDO:0005303,cocaine|cocaine dependence|drug dependence
2,NCT00000428,fibromyalgia,amitriptyline|amitriptyline plus fluoxitine|fl...,Neuromuscular Diseases,Amitriptyline|amitriptyline plus fluoxitine|fl...,C0002600|-1|-1|C0016365,fibromyalgia,MONDO:0005546,-1,-1,MONDO:0005546,fibromyalgia
3,NCT00000439,alcohol dependence|bipolar alcoholics|bipolar ...,depacon|sodium valproate|valproate|valproic acid,Psychiatry and Psychology Category,depacon|Valproate Sodium|Valproate|Valproic Acid,-1|C0037567|C0080356|C0042291,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1,-1|-1,-1|-1,MONDO:0002046|-1|MONDO:0004985|-1,alcohol abuse|bipolar alcoholics|bipolar disor...
4,NCT00001956,hyperalgesia|oral surgery|pain|removal of thir...,|ketorolac|lidocaine|midazolam|sedative,Neurologic Manifestations,Ketorolac|Lidocaine|Midazolam|sedative,C0073631|C0023660|C0026056|-1,hyperalgesia|oral surgery|obsolete disorder in...,-1|-1|MONDO:0021668|-1,-1,-1,-1|-1|MONDO:0021668|-1,hyperalgesia|oral surgery|obsolete disorder in...
...,...,...,...,...,...,...,...,...,...,...,...,...
18604,NCT06282640,carpal tunnel syndrome|cts|pain,ergocalciferol|v|vitamin d,Neuromuscular Diseases,Ergocalciferol|v|VITAMIN D,C0014695|-1|C3714503,carpal tunnel syndrome|obsolete disorder invol...,MONDO:0007275|MONDO:0021668,MONDO:0003615|-1,nerve compression syndrome|-1,MONDO:0007275|MONDO:0021668|MONDO:0003615,carpal tunnel syndrome|obsolete disorder invol...
18605,NCT06287502,osteoporosis|sarcopenia,hydroxy β-methylbutyrate|β-hydroxy β-methylbut...,Neurologic Manifestations,HYDROXY-BETA-METHYLBUTYRATE|HYDROXY-BETA-METHY...,C1995592|C1995592|C1995592,osteoporosis|obsolete sarcopenia,MONDO:0005298|MONDO:0006516,MONDO:0019019|-1,osteogenesis imperfecta|-1,MONDO:0005298|MONDO:0006516|MONDO:0019019,osteoporosis|obsolete sarcopenia|osteogenesis ...
18606,NCT06289335,cesarean section|emergence delirium|post-anest...,dexmedetomidine|ondansetron|saline solution,Neurologic Manifestations,Dexmedetomidine|Ondansetron|Saline,C0113293|C0061851|C0036082,cesarean section|delirium|post-anesthetic shiv...,-1|MONDO:0045057|-1|-1,-1,-1,-1|MONDO:0045057|-1|-1,cesarean section|delirium|post-anesthetic shiv...
18607,NCT06292351,alzheimer disease|dementia,dmb-i|dmb-i (dimebon),Neurodegenerative Diseases,dmb-i|dmb-i (dimebon),-1|-1,Alzheimer disease|dementia,MONDO:0004975|MONDO:0001627,MONDO:0005574|-1,tauopathy|-1,MONDO:0004975|MONDO:0001627|MONDO:0005574,Alzheimer disease|dementia|tauopathy


In [385]:
df_final_preclinical.to_csv("./data/mapped_all/mapped_preclinical_data_with_mondo_parents.csv", index=False)
df_final_clinical.to_csv("./data/mapped_all/mapped_clinical_data_with_mondo_parents.csv", index=False)


In [386]:
df_final_preclinical[df_final_preclinical['PMID']=="29526407"]

,PMID,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,linkbert_mapped_conditions,linkbert_mapped_drugs,disease_term_mondo_norm,disease_mondo_termid,drug_term_umls_norm,drug_umls_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label
545923,29526407,multiple sclerosis,murine anti-cd52 antibody|anti-cd52 antibody|a...,multiple sclerosis,murine anti-cd52 antibody|anti-cd52 antibody|a...,multiple sclerosis,MONDO:0005301,murine anti-cd52 antibody|anti-cd52 antibody|a...,-1|-1|-1,-1,-1,MONDO:0005301,multiple sclerosis
